# Train + evaluate the CodeBERT defect predictor
**Before running:** Runtime → Change runtime type → **T4 GPU**. Upload `code-review-ai.zip` to `/content` (rename it so it has no spaces/brackets, e.g. not `code-review-ai (1).zip`).

In [ ]:
!nvidia-smi | head -n 12

In [ ]:
!rm -rf /content/code-review-ai
!unzip -q -o /content/code-review-ai.zip -d /content
!ls /content/code-review-ai/ml

In [ ]:
!pip -q install -r /content/code-review-ai/ml/requirements.txt

## 1. Datasets (absolute paths, so no `cd` problems)

In [ ]:
!rm -rf /content/data && mkdir -p /content/data
!git clone -q --depth 1 https://github.com/soarsmu/BugsInPy /content/data/BugsInPy
!git clone -q --depth 1 https://github.com/rjust/defects4j /content/data/defects4j
!ls /content/data/BugsInPy/projects | head -3
!ls /content/data/defects4j/framework/projects | head -3

## 2. Build labelled examples (expect ~6,000 samples, never 0)

In [ ]:
!cd /content/code-review-ai/ml && python data_prep.py --bugsinpy /content/data/BugsInPy --defects4j /content/data/defects4j --out /content/data/processed

## 3. Train (about 15–30 min on a T4)

In [ ]:
!cd /content/code-review-ai/ml && python train.py --data /content/data/processed --out /content/artifacts/codebert-defect --epochs 4 --batch 16

## 4. Evaluate honestly: baselines, 95% CIs, and a threshold picked on the validation set
Writes `results.md`, `results.json`, `curves.png` and **`threshold.json`** (the backend reads this automatically).

In [ ]:
!cd /content/code-review-ai/ml && python evaluate.py --data /content/data/processed --model /content/artifacts/codebert-defect --out /content/artifacts/codebert-defect

## 5. Optional: one experiment for your ablation table
Each run goes to its own folder; evaluate each and compare the AUC rows. Change one thing at a time.

In [ ]:
# Example: longer training with a higher learning rate
!cd /content/code-review-ai/ml && python train.py --data /content/data/processed --out /content/artifacts/exp_e8_lr3e5 --epochs 8 --lr 3e-5 --batch 16
!cd /content/code-review-ai/ml && python evaluate.py --data /content/data/processed --model /content/artifacts/exp_e8_lr3e5 --out /content/artifacts/exp_e8_lr3e5

## 6. Download the model (includes threshold.json and the results files)

In [ ]:
!cd /content/artifacts && rm -rf codebert-defect/checkpoints && zip -qr /content/codebert-defect.zip codebert-defect
from google.colab import files
files.download('/content/codebert-defect.zip')

## 7. Optional: publish to the Hugging Face Hub (needed for Render)
Create a **write** token at huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import login, HfApi
login()  # paste your token
api = HfApi()
repo = 'YOUR_USERNAME/codebert-defect'   # <-- change this
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path='/content/artifacts/codebert-defect', repo_id=repo, ignore_patterns=['checkpoints/*'])
print('uploaded to', repo)